In [1]:
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#       jupytext_version: 1.19.6
#   kernelspec:
#     display_name: Python (latest Jaspy)
#     language: python
#     name: jaspy
# ---

# NB 02 Binary conditional copulas (per fold)

For each blocked fold the divergence w is split at the **fold-training median** into a low and a high regime. In each regime seven parametric families (Gaussian, t, Clayton, Gumbel, Gumbel (180°), Frank, Joe) and a CODINE net (the two per-regime nets of the split CODINE; 3 seeds) are fitted on that fold's training steps of the regime. The pseudo-observations are the **kernel-conditional** ones of notebook 00: the marginals are already conditional on w, and the binary split conditions only the copula. The 49 low/high pairs are built by indexing; the pair with the lowest training AIC is the data-driven binary model, chosen per fold.

Per fold and regime: training AIC, test CVPE inside the regime, marginal-uniformity KS distance D_max (D_max <= 0.07 per regime component; a pair is admissible in a fold only if both of its regime components pass). Each table is written as LaTeX and CSV.

## Imports and per-fold inputs

In [2]:
import os
os.environ["OMP_NUM_THREADS"] = "1"; os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"; os.environ["NUMEXPR_NUM_THREADS"] = "1"
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd

from run_support import load_artifact, save_artifact, SMOKE, MAX_WORKERS   # production defaults; override with THESIS_SMOKE / THESIS_MAX_WORKERS (see run_support.py)
import gate_config as GC
from copula_families import signed_params_of, branch_of, fit_and_evaluate, DEFAULT_FAMILIES   # signed primary parameter; branch (own/turned rotation) of a fit
# Unconditional models / penalty calibration do not use previously calibrated network weights.
os.environ["THESIS_ALLOW_UNCALIBRATED_IMPORT"] = "1"
import fold_pipeline as FP
import reporting_tables
reporting_tables.configure("02_binary_conditional_copulas")

FOLDS = FP.FOLDS
ARR = FP.load_all_folds()          # per-fold arrays: kernel-conditional u1/u2, masks, b, w
print("kernel bandwidth h by fold =", load_artifact("preprocessing_folds_summary").get('kernel_h_by_fold'))

# Fits, marginal KS checks and losses of the 2 x 7 regime families (shared with notebooks 01 and 04 through the
# resume files in artifacts/fold_batch: whichever notebook runs first computes them).
PARAM, _ = FP.run_param_stage(ARR, max_workers=MAX_WORKERS)
# The two regime CODINEs: 4 folds x 3 seeds x 2; marginal KS check on seed 0 only.
NEURAL = FP.run_batch(FP.build_fold_jobs(ARR, None, smoke=SMOKE, kinds=("codine_regime",)),
                      max_workers=MAX_WORKERS, batch_dir=FP.BATCH_DIR, desc='regime CODINE')

kernel bandwidth h by fold = {'A1': 0.22832144093183465, 'A2': 0.6636094179528018, 'B1': 0.6643855353375984, 'B2': 0.6643855353375984}
parametric folds: 8 valid resume results, 0 assigned jobs to run


regime CODINE: 0 valid resume results, 24 assigned jobs to run


regime CODINE: 13 workers (CPU/task limit 24, estimated 1.5 GiB each; RAM headroom checked at pool creation)


regime CODINE:   0%|          | 0/24 [00:00<?, ?it/s]

## Families per regime and fold

In [3]:
REGS = ('low', 'high')
RES, TABLES = {}, {}
for f in FOLDS:
    A_ = ARR[f]; tr, te, hi = A_['train'], A_['test'], A_['high']
    RES[f], TABLES[f] = {}, {}
    for reg in REGS:
        rm = hi if reg == 'high' else ~hi
        _, rows = fit_and_evaluate(A_['u1'], A_['u2'], train_mask=tr & rm, test_mask=te & rm,
                                   families=list(DEFAULT_FAMILIES), quiet=True)
        RES[f][reg] = rows
        tab = pd.DataFrame([dict(Family=r['name'], Branch=branch_of(r['cop']), AIC_train=r['aic'], CVPE_test_regime=r['cvpe'], KS_D=PARAM[f].get('ks', {}).get((reg, r['name']), np.nan)) for r in rows])
        tab['Admissible'] = [GC.ks_seed_pass(d) for d in tab.KS_D]
        tab.loc[len(tab)] = dict(Family='CODINE', Branch='-', AIC_train=np.nan, CVPE_test_regime=np.nan, KS_D=max((NEURAL.get(FP.net_key(f, seed, f'{FP.SPLIT_CODINE}|{reg}'), {}).get('ks', np.nan) for seed in FP.SEEDS), default=np.nan), Admissible=all(GC.ks_seed_pass(NEURAL.get(FP.net_key(f, seed, f'{FP.SPLIT_CODINE}|{reg}'), {}).get('ks')) for seed in FP.SEEDS))
        TABLES[f][reg] = tab
    print(f"\n=== fold {f}: train n={int(tr.sum())}, test n={int(te.sum())}; training steps low/high = "
          f"{int((tr & ~hi).sum())}/{int((tr & hi).sum())}; AIC-selected pair (low | high) = {PARAM[f]['selected_pair']}")
    for reg in REGS:
        print(f'{reg} divergence:'); display(TABLES[f][reg])


=== fold A1: train n=1394, test n=1393; training steps low/high = 697/697; AIC-selected pair (low | high) = ('Gumbel', 'Gaussian')
low divergence:


,Family,Branch,AIC_train,CVPE_test_regime,KS_D,Admissible
0,Gaussian,-,-18.559075,0.160741,0.023433,True
1,t,-,-17.660257,0.160734,0.023760,True
2,Clayton,own,-9.501024,0.160879,0.022518,True
3,Gumbel,own,-19.957427,0.159957,0.025600,True
4,Gumbel (180°),own,-11.454771,0.160903,0.024896,True
5,Frank,-,-18.184580,0.160913,0.022254,True
6,Joe,own,-16.101296,0.159391,0.026808,True
7,CODINE,-,NaN,NaN,0.044067,True


high divergence:


,Family,Branch,AIC_train,CVPE_test_regime,KS_D,Admissible
0,Gaussian,-,-7.468140,0.141946,0.024410,True
1,t,-,-4.879468,0.141893,0.023878,True
2,Clayton,own,-4.943025,0.141278,0.022576,True
3,Gumbel,own,-6.547981,0.141368,0.024972,True
4,Gumbel (180°),own,-2.004271,0.140848,0.024932,True
5,Frank,-,-5.306729,0.141404,0.022741,True
6,Joe,own,-4.489587,0.140814,0.026719,True
7,CODINE,-,NaN,NaN,0.049754,True



=== fold A2: train n=1393, test n=1394; training steps low/high = 696/697; AIC-selected pair (low | high) = ('t', 'Gumbel (180°)')
low divergence:


,Family,Branch,AIC_train,CVPE_test_regime,KS_D,Admissible
0,Gaussian,-,-5.069817,0.169977,0.022370,True
1,t,-,-14.095085,0.170173,0.024966,True
2,Clayton,own,-2.036543,0.171445,0.024249,True
3,Gumbel,own,-12.615187,0.169698,0.025450,True
4,Gumbel (180°),own,-5.893698,0.170663,0.023687,True
5,Frank,-,-3.649854,0.170285,0.022433,True
6,Joe,own,-13.033721,0.170351,0.027808,True
7,CODINE,-,NaN,NaN,0.069815,True


... (6 outputs left out)


=== fold B2: train n=1345, test n=1345; training steps low/high = 672/673; AIC-selected pair (low | high) = ('t', 'Gumbel')
low divergence:


,Family,Branch,AIC_train,CVPE_test_regime,KS_D,Admissible
0,Gaussian,-,-2.851086,0.164607,0.022491,True
1,t,-,-9.176872,0.164705,0.023335,True
2,Clayton,own,-4.070672,0.165865,0.022840,True
3,Gumbel,own,-3.450364,0.164213,0.023520,True
4,Gumbel (180°),own,-7.565020,0.165276,0.022696,True
5,Frank,-,-2.373814,0.164814,0.022622,True
6,Joe,own,-2.064719,0.164901,0.023291,True
7,CODINE,-,NaN,NaN,0.038300,True


high divergence:


,Family,Branch,AIC_train,CVPE_test_regime,KS_D,Admissible
0,Gaussian,-,-7.806316,0.174290,0.024295,True
1,t,-,-8.138424,0.174394,0.023854,True
2,Clayton,own,-2.302593,0.172568,0.023794,True
3,Gumbel,own,-11.863005,0.175232,0.026246,True
4,Gumbel (180°),own,-2.647510,0.172788,0.024152,True
5,Frank,-,-8.119997,0.174633,0.024094,True
6,Joe,own,-10.708634,0.175168,0.027392,True
7,CODINE,-,NaN,NaN,0.050854,True


## Write the per-fold tables

In [4]:
for f in FOLDS:
    for reg in REGS:
        reporting_tables.write_table(TABLES[f][reg].set_index('Family'), f'binary_copulas_fold_{f}_{reg}',
            f'Binary-split copulas, fold {f}, {reg}-divergence regime: training AIC, CVPE on the regime test steps and marginal-uniformity KS distance (Dmax <= {GC.KS_D_MAX} per regime component).',
            label=f'tab:binary_{f}_{reg}', col_format='llrrrc')

## AIC-selected pair per fold

In [5]:
sel = pd.DataFrame({f: dict(zip(REGS, PARAM[f]['selected_pair'])) for f in FOLDS})
reporting_tables.show_table(sel, 'aic_selected_pairs', caption='AIC-selected family pair (low, high divergence) per fold.')

**aic selected pairs**

,A1,A2,B1,B2
low,Gumbel,t,Gumbel,t
high,Gaussian,Gumbel (180°),Clayton,Gumbel


,A1,A2,B1,B2
low,Gumbel,t,Gumbel,t
high,Gaussian,Gumbel (180°),Clayton,Gumbel


## Save

In [6]:
save_artifact("binary_divergence_copulas_folds",
              tables=TABLES, folds=list(FOLDS), selected_pairs=sel,
              family_params={f: {reg: {r['name']: signed_params_of(r['cop']) for r in RES[f][reg]} for reg in REGS} for f in FOLDS})
print("saved binary_divergence_copulas_folds")

saved binary_divergence_copulas_folds


## Table layout

Every table with a tabular is wrapped so that it shrinks to the text width when it is wider and is never enlarged (`reporting_tables.fit_all_tables`); no table needs a manual size edit in the report.

In [7]:
import reporting_tables
print('table files fitted to the text width:', reporting_tables.fit_all_tables())

table files fitted to the text width: []
